In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('zoomcamp_yellow_taxi') \
    .config("spark.driver.extraJavaOptions", 
            "--add-opens=java.base/java.lang=ALL-UNNAMED "
            "--add-opens=java.base/java.nio=ALL-UNNAMED "
            "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED") \
    .getOrCreate()

print("Spark session created successfully!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 16:04:18 WARN Utils: Your hostname, codespaces-62d77b, resolves to a loopback address: 127.0.0.1; using 10.0.0.163 instead (on interface eth0)
26/03/10 16:04:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 16:04:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session created successfully!


In [2]:
spark.version

'4.1.1'

In [4]:
# Download the data 
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-10 15:54:41--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.173.143, 52.85.173.7, 52.85.173.65, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.173.143|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  93.7MB/s    in 0.7s    

2026-03-10 15:54:42 (93.7 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [3]:
# Read the data
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [4]:
# Partition the data
df_yellow.repartition(4).write.mode('overwrite').parquet('output/yellow/2025/11')


In [5]:
# Check the average file size
import os
files = [f for f in os.listdir('output/yellow/2025/11/') if f.endswith('.parquet')]
sizes = [os.path.getsize(f'output/yellow/2025/11/{f}') / (1024 * 1024) for f in files]
print(f"Average file size: {sum(sizes) / len(sizes):.2f} MB")

Average file size: 25.33 MB


In [7]:
# Question 3, number of trips on 15th November 
from pyspark.sql import functions as F
trips_15 = df_yellow.filter(F.to_date(df_yellow.tpep_pickup_datetime) == '2025-11-15').count()
print(f"{trips_15} on November 15th")

162604 on November 15th


In [19]:
# Longest trip 
from pyspark.sql.functions import timestamp_diff

# Calculate duration in seconds directly
df_yellow = df_yellow.withColumn("duration_hours", 
    (timestamp_diff("SECOND", col("tpep_pickup_datetime"), col("tpep_dropoff_datetime"))/3600)
)

In [20]:
df_yellow.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|duration_seconds|      duration_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+--------------------+
|       7| 2025-

In [21]:
from pyspark.sql.functions import col, desc

df_yellow.orderBy(desc("duration_hours")).limit(1).show()




+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|duration_seconds|   duration_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+-----------------+
|       2| 2025-11-26 20:

In [22]:
# Download Zone Lookup
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

# Read Zones
df_zones = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')

# Create Temporary Views for SQL join
df_yellow.createOrReplaceTempView('yellow_data')
df_zones.createOrReplaceTempView('zones')

# Join and group to find the least frequent zone
least_frequent = spark.sql("""
    SELECT
        z.Zone,
        COUNT(1) as count
    FROM
        yellow_data y
    JOIN
        zones z ON y.PULocationID = z.LocationID
    GROUP BY
        z.Zone
    ORDER BY
        count ASC
    LIMIT 5
""")

least_frequent.show()

--2026-03-10 16:42:23--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.146, 18.239.115.213, 18.239.115.86, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.146|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-10 16:42:23 (1.00 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+

